# Notebook 05: Unsupervised Job Profile and Skill Clustering
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Discover latent job archetypes and skill groupings without ground-truth labels.
2. Input text: composite representation (`job_title + job_description + requirements`).
3. Embeddings: SBERT Multilingual dense vectors (384 dimensions).
4. Run K-Means sweep on full dataset ($k \in [3, 4, 5, 6, 8, 10]$).
5. Fair head-to-head comparison on standardized sample (N=3,000) across K-Means, Agglomerative, DBSCAN, and GMM.
6. Evaluate using Silhouette Score (Cosine), Davies-Bouldin Index, Calinski-Harabasz Score, ARI, and NMI.
7. Extract technical keywords with Vietnamese + English stopword removal and assign distinct, non-colliding cluster profiles.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.models.clustering import run_clustering_experiments

## 1. Load Clustering Dataset and Composite Embeddings

In [ ]:
df_cluster = pd.read_csv(config.PROCESSED_DATA_DIR / "clustering_data.csv")
cluster_embeddings = np.load(config.EMBEDDINGS_DIR / "sbert_clustering_embeddings.npy")

print(f"Clustering samples: {df_cluster.shape[0]}, Embedding dimension: {cluster_embeddings.shape[1]}")

## 2. Train Clustering Algorithms & Fair Head-to-Head Benchmark
This cell executes live training of K-Means (k=8), Agglomerative, GMM, and DBSCAN on standardized sample.

In [ ]:
# Execute live clustering experiments
clustering_results, assignments, best_km_model, cluster_meta = run_clustering_experiments(
    df_cluster,
    cluster_embeddings,
    df_cluster['composite_text'].tolist()
)

print("\n=== Clustering Benchmark Results (Fair Comparison Table) ===")
display(clustering_results)

## 3. Discovered Cluster Profiles and Distinct Keywords

In [ ]:
kw_path = config.REPORTS_DIR / "tables" / "cluster_keywords.csv"
if kw_path.exists():
    kw_df = pd.read_csv(kw_path)
    print("=== Discovered Cluster Profiles & Technical Keywords ===")
    display(kw_df)

## 4. Clustering Visualizations

In [ ]:
for plot_name in [
    'clustering_elbow_curve.png',
    'clustering_silhouette_comparison.png',
    'clustering_pca_clusters.png',
    'clustering_distribution.png'
]:
    p = config.REPORTS_DIR / 'figures' / plot_name
    if p.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(p)))